In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip multilabel-classification-dataset.zip -d /content/dataset

In [ ]:
#  Step 1: Load & Prepare Data
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

df = pd.read_csv(f"{shivanandmn_multilabel_classification_dataset_path}/train.csv")
label_cols = ['Computer Science', 'Physics', 'Mathematics', 'Statistics',
              'Quantitative Biology', 'Quantitative Finance']
X = (df["TITLE"] + " " + df["ABSTRACT"]).tolist()
y = torch.tensor(df[label_cols].values, dtype=torch.float32)

In [ ]:
#  Step 2: Tokenization (RoBERTa)
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("roberta-base")
encodings = tokenizer(X, truncation=True, padding=True, max_length=512, return_tensors="pt")


In [ ]:
#  Step 3: Train-Validation Split
t_input_ids = encodings["input_ids"]
t_attention_mask = encodings["attention_mask"]

train_idx, val_idx = train_test_split(np.arange(len(df)), test_size=0.2, random_state=42)

train_dataset = TensorDataset(t_input_ids[train_idx], t_attention_mask[train_idx], y[train_idx])
val_dataset = TensorDataset(t_input_ids[val_idx], t_attention_mask[val_idx], y[val_idx])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4)


In [ ]:
#  Step 4: Define RoBERTa Model
from transformers import AutoModelForSequenceClassification
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=y.shape[1]).to(device)



In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np
import torch.nn as nn
import torch.optim as optim

# Define optimizer and loss BEFORE training
optimizer = optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.BCEWithLogitsLoss()

epochs = 10

for epoch in range(epochs):

    # ================= TRAIN =================
    model.train()
    train_loss = 0

    train_preds = []
    train_targets = []

    for batch in train_loader:
        ids, mask, targets = [b.to(device) for b in batch]

        optimizer.zero_grad()
        outputs = model(input_ids=ids, attention_mask=mask).logits
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        probs = torch.sigmoid(outputs)
        preds = (probs >= 0.5).float()

        train_preds.append(preds.detach().cpu())
        train_targets.append(targets.detach().cpu())

    train_preds = torch.cat(train_preds).numpy()
    train_targets = torch.cat(train_targets).numpy()

    avg_train_loss = train_loss / len(train_loader)

    train_acc = (train_preds == train_targets).mean()
    train_prec = precision_score(train_targets, train_preds, average="micro", zero_division=0)
    train_rec = recall_score(train_targets, train_preds, average="micro", zero_division=0)
    train_f1 = f1_score(train_targets, train_preds, average="micro", zero_division=0)

    # ================= VALIDATION =================
    model.eval()
    val_loss = 0

    val_preds = []
    val_targets = []

    with torch.no_grad():
        for batch in val_loader:
            ids, mask, targets = [b.to(device) for b in batch]

            outputs = model(input_ids=ids, attention_mask=mask).logits
            loss = criterion(outputs, targets)
            val_loss += loss.item()

            probs = torch.sigmoid(outputs)
            preds = (probs >= 0.5).float()

            val_preds.append(preds.cpu())
            val_targets.append(targets.cpu())

    val_preds = torch.cat(val_preds).numpy()
    val_targets = torch.cat(val_targets).numpy()

    avg_val_loss = val_loss / len(val_loader)

    val_acc = (val_preds == val_targets).mean()
    val_prec = precision_score(val_targets, val_preds, average="micro", zero_division=0)
    val_rec = recall_score(val_targets, val_preds, average="micro", zero_division=0)
    val_f1 = f1_score(val_targets, val_preds, average="micro", zero_division=0)

    # ================= PRINT =================
    print(
        f"Epoch {epoch+1}/{epochs}\n"
        f"Train | Loss: {avg_train_loss:.4f} | Acc: {train_acc:.4f} | "
        f"Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f}\n"
        f"Val   | Loss: {avg_val_loss:.4f} | Acc: {val_acc:.4f} | "
        f"Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f}\n"
    )

    # ================= SAVE =================
    torch.save(model.state_dict(), f"deberta_epoch{epoch+1}.pth")

In [ ]:
#  Step 6: Validation Evaluation
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, hamming_loss

model.eval()
val_preds, val_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        ids, mask, targets = [b.to(device) for b in batch]
        outputs = model(input_ids=ids, attention_mask=mask).logits
        val_preds.append(torch.sigmoid(outputs).cpu())
        val_labels.append(targets.cpu())

preds = torch.cat(val_preds).numpy()
true = torch.cat(val_labels).numpy()
binary_preds = (preds > 0.4).astype(int)

print("\n RoBERTa (TITLE + ABSTRACT) Validation Metrics")
print("Accuracy Score:", accuracy_score(true, binary_preds))
print("Micro F1 Score:", f1_score(true, binary_preds, average='micro'))
print("Macro F1 Score:", f1_score(true, binary_preds, average='macro'))
print("Micro Precision:", precision_score(true, binary_preds, average='micro'))
print("Micro Recall:", recall_score(true, binary_preds, average='micro'))
print("Hamming Loss:", hamming_loss(true, binary_preds))


In [ ]:
#  Step 7: Per-Label Accuracy (Validation)
print("\n Per-label Accuracy (Validation)")
accuracies = []
for i, label in enumerate(label_cols):
    acc = np.sum(true[:, i] == binary_preds[:, i]) / len(true[:, i])
    print((label, np.float64(acc)))
    accuracies.append(acc)
print("Average accuracy =", np.mean(accuracies))

In [ ]:

#  Step 10: Save Model
model_save_path = "roberta_title_abstract_model.pt"
torch.save(model.state_dict(), model_save_path)
print("\n Model saved to:", model_save_path)